In [2]:
# file: baseline_ngram_improved_fixed.py
from __future__ import annotations
import math, unicodedata, sys, json, random
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Any
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Use sacrebleu for trustworthy, comparable metrics
import sacrebleu


# ---------------- Path helpers ---------------- #

LIKELY_DIRS = [
    ".", "data", "dataset", "datasets", "inputs", "input",
    "/mnt/data", "/kaggle/input", "/workspace", "/content",
]

def resolve_one(path_or_name: str) -> Path:
    p = Path(path_or_name)
    if p.exists():
        return p.resolve()
    name = p.name
    for d in LIKELY_DIRS:
        cand = Path(d) / name
        if cand.exists():
            return cand.resolve()
    raise FileNotFoundError(f"Could not find '{path_or_name}'.")

def resolve_triplet(lang: str) -> tuple[Path, Path, Path]:
    return (
        resolve_one(f"{lang}_train.csv"),
        resolve_one(f"{lang}_val.csv"),
        resolve_one(f"{lang}_test.csv"),
    )


# ---------------- Normalization ---------------- #

def normalize_text(s: str, lowercase: bool = True, strip_accents: bool = True) -> str:
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", str(s))
    if strip_accents:
        s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    if lowercase:
        s = s.lower()
    return " ".join(s.split())


# ---------------- Metrics (sacrebleu) ---------------- #

def corpus_scores(preds: List[str], refs: List[str]) -> Dict[str, float]:
    bleu = sacrebleu.corpus_bleu(preds, [refs], tokenize="13a").score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    return {"BLEU": bleu, "BLEU_0_1": bleu / 100.0, "chrF": chrf,
            "ExactMatch": 100.0 * sum(p.strip() == r.strip() for p, r in zip(preds, refs)) / max(1, len(refs))}


# ---------------- Simple BM25 (optional) ---------------- #

class BM25:
    def __init__(self, docs: List[List[str]], k1: float = 1.2, b: float = 0.75):
        self.k1, self.b, self.docs = k1, b, docs
        self.N = len(docs)
        self.avgdl = np.mean([len(d) for d in docs]) if docs else 0.0
        self.df: Dict[str, int] = {}
        self.freq: List[Dict[str, int]] = []
        for d in docs:
            c: Dict[str, int] = {}
            for t in d:
                c[t] = c.get(t, 0) + 1
            self.freq.append(c)
            for t in c:
                self.df[t] = self.df.get(t, 0) + 1
        self.idf = {t: math.log(1 + (self.N - df + 0.5) / (df + 0.5)) for t, df in self.df.items()}

    def scores(self, q: List[str]) -> np.ndarray:
        out = np.zeros(self.N, dtype=float)
        for i in range(self.N):
            L = len(self.docs[i]) or 1
            s = 0.0
            for t in q:
                tf = self.freq[i].get(t, 0)
                if tf == 0:
                    continue
                idf = self.idf.get(t, 0.0)
                s += idf * ((tf * (self.k1 + 1)) / (tf + self.k1 * (1 - self.b + self.b * L / (self.avgdl or 1))))
            out[i] = s
        return out


# ---------------- Retriever ---------------- #

@dataclass
class RetrievalConfig:
    char_ngram_range: Tuple[int, int] = (3, 5)  # crucial for codemix
    max_features: int = 300_000                 # larger = better coverage
    knn_k: int = 50
    rerank_top_k: int = 10
    length_sigma: float = 30.0
    use_bm25: bool = True
    bm25_k1: float = 1.2
    bm25_b: float = 0.75
    bm25_weight: float = 0.25   # modest fusion with TF-IDF


class RetrievalModel:
    """
    1) Index on TRAIN **sources** (codemix). 2) For a query source, find nearest train sources.
    3) Return the corresponding TRAIN **targets** (English) with a simple rerank.
    Why: classic retrieval baseline for translation parallels your Hinglish setup.
    """
    def __init__(self, cfg: RetrievalConfig):
        self.cfg = cfg
        self.vec = TfidfVectorizer(
            analyzer="char",
            ngram_range=cfg.char_ngram_range,
            lowercase=False,
            min_df=1,
            max_features=cfg.max_features,
        )
        self.nn: Optional[NearestNeighbors] = None
        self.train_src_norm: List[str] = []
        self.train_src_raw: List[str] = []
        self.train_tgt: List[str] = []
        self.bm25: Optional[BM25] = None

    def fit(self, train_src: List[str], train_tgt: List[str]) -> "RetrievalModel":
        self.train_src_raw = list(train_src)
        self.train_src_norm = [normalize_text(s) for s in train_src]
        self.train_tgt = list(train_tgt)
        X = self.vec.fit_transform(self.train_src_norm)
        self.nn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=min(self.cfg.knn_k, max(1, len(self.train_src_norm))))
        self.nn.fit(X)
        if self.cfg.use_bm25:
            self.bm25 = BM25([s.split() for s in self.train_src_norm], k1=self.cfg.bm25_k1, b=self.cfg.bm25_b)
        return self

    def _predict_one(self, src: str) -> str:
        q = normalize_text(src)
        Xq = self.vec.transform([q])
        dist, idx = self.nn.kneighbors(Xq, n_neighbors=min(self.cfg.knn_k, len(self.train_src_norm)))
        idx = idx[0].astype(int)
        tfidf_sim = 1.0 - dist[0]  # cosine similarity
        # Optional BM25 fusion on sources (same space)
        if self.bm25 is not None:
            bm = self.bm25.scores(q.split())
            bm_cand = bm[idx]
            if bm_cand.max() > 0:
                bm_cand = bm_cand / bm_cand.max()
            fused = 0.75 * tfidf_sim + self.cfg.bm25_weight * bm_cand
        else:
            fused = tfidf_sim

        top_k = min(self.cfg.rerank_top_k, len(idx))
        sel = np.argsort(fused)[-top_k:][::-1]
        cand_ids = idx[sel]
        cand_scores = fused[sel]

        # mild length-aware rerank
        q_len = max(1, len(src.split()))
        cand_txt = [self.train_tgt[i] for i in cand_ids]
        cand_len = np.array([max(1, len(t.split())) for t in cand_txt], dtype=float)
        len_pen = np.exp(-np.abs(cand_len - q_len) / self.cfg.length_sigma)
        final = cand_scores * len_pen
        best = int(np.argmax(final))
        return cand_txt[best]

    def predict(self, sources: List[str]) -> List[str]:
        return [self._predict_one(s) for s in sources]


# ---------------- Audit & Oracle ---------------- #

def ascii_ratio(s: str) -> float:
    if not s:
        return 1.0
    return sum(1 for ch in s if ord(ch) < 128) / len(s)

def audit_split(tag: str, src: List[str], ref: List[str], pred: List[str]) -> Dict[str, Any]:
    n = len(pred)
    eq_src = sum(1 for a, b in zip(pred, src) if a == b)
    empties = sum(1 for a in pred if len(a.strip()) == 0)
    dup_top = pd.Series(pred).value_counts().iloc[0] if n else 0
    ascii_avg = np.mean([ascii_ratio(p) for p in pred]) if n else 1.0
    m = corpus_scores(pred, ref)
    print(f"[{tag}] n={n} | BLEU={m['BLEU']:.2f} chrF={m['chrF']:.2f} EM={m['ExactMatch']:.2f} | pred==src={eq_src} | empty={empties} | topdup={dup_top} | ascii={ascii_avg:.3f}")
    flags = []
    if m["BLEU"] < 2.0 and m["chrF"] > 20.0:
        flags.append("BLEU≈0 but chrF>20 → often wrong column / copied source / misalignment.")
    if eq_src / max(1, n) >= 0.01:
        flags.append("≥1% predictions equal SOURCE (wrong column).")
    if empties > 0:
        flags.append("Empty predictions present.")
    if dup_top / max(1, n) > 0.1:
        flags.append("High duplication in predictions.")
    if ascii_avg < 0.85:
        flags.append("Preds contain many non-ASCII chars; expected English.")
    if flags:
        print("  FLAGS:")
        for f in flags:
            print("   -", f)
    return {**m, "eq_src": eq_src, "empties": empties, "dup_top": int(dup_top), "ascii_avg": float(ascii_avg), "flags": flags}

def oracle_upper_bound(train_refs: List[str], test_refs: List[str]) -> float:
    """Best train reference for each test reference (not a real system; an upper bound for retrieval)."""
    vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), lowercase=True, min_df=1)
    X_tr = vec.fit_transform(train_refs)
    X_te = vec.transform(test_refs)
    nn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=1).fit(X_tr)
    dist, idx = nn.kneighbors(X_te, n_neighbors=1)
    best = [train_refs[i] for i in idx[:, 0]]
    return sacrebleu.corpus_bleu(best, [test_refs], tokenize="13a").score


# ---------------- Pipeline ---------------- #

def run_language(lang: str, train_path: Optional[str], val_path: Optional[str], test_path: Optional[str],
                 outdir: str, cfg: RetrievalConfig) -> pd.DataFrame:
    if train_path and val_path and test_path:
        tr_p, va_p, te_p = Path(train_path), Path(val_path), Path(test_path)
    else:
        tr_p, va_p, te_p = resolve_triplet(lang)

    out_dir = Path(outdir); out_dir.mkdir(parents=True, exist_ok=True)

    # Load & canonicalize columns
    tr = pd.read_csv(tr_p)
    va = pd.read_csv(va_p)
    te = pd.read_csv(te_p)
    for df in (tr, va, te):
        assert "source" in df.columns and "target" in df.columns, f"{lang}: CSV must have columns ['source','target']"
        df["source"] = df["source"].astype(str).fillna("")
        df["target"] = df["target"].astype(str).fillna("")

    # Fit per language on TRAIN **sources** → return TRAIN **targets**
    model = RetrievalModel(cfg).fit(tr["source"].tolist(), tr["target"].tolist())

    def eval_split(df: pd.DataFrame, split: str) -> Dict[str, Any]:
        src = df["source"].tolist()
        refs = df["target"].tolist()
        preds = model.predict(src)
        out_path = out_dir / f"{lang}_{split}_baselinePLUS_preds.csv"
        pd.DataFrame({"source": df["source"], "reference": refs, "prediction": preds}).to_csv(out_path, index=False)
        audit = audit_split(f"{lang}-{split}", src, refs, preds)
        return {"lang": lang, "split": split, **audit, "pred_path": str(out_path)}

    rows = [eval_split(va, "val"), eval_split(te, "test")]

    # Oracle upper bound (helps decide if retrieval is viable on this data)
    ub = oracle_upper_bound(tr["target"].tolist(), te["target"].tolist())
    (out_dir / f"{lang}_oracle_upper_bound.txt").write_text(f"{ub:.2f}\n", encoding="utf-8")
    print(f"[{lang}] Oracle upper bound BLEU (train-ref -> test-ref): {ub:.2f}")

    # Fail fast for Spanglish if flags indicate a broken pipe
    if lang.lower().startswith("spanglish"):
        any_flags = any(r["flags"] for r in rows)
        if any_flags:
            print("\n[VERDICT] ❌ Spanglish baseline is inconsistent. See FLAGS above.\n"
                  "Fix order/columns or per-language fit; predictions must be English.")
            # Exit non-zero in scripts/CI; comment next line if running interactively
            # sys.exit(1)

    return pd.DataFrame(rows)


def main(argv: List[str] | None = None):
    argv = argv or sys.argv[1:]
    outdir = "outputs_baseline"
    if "--out" in argv:
        i = argv.index("--out"); outdir = argv[i+1]

    cfg = RetrievalConfig()  # defaults tuned for code-mix

    res = []
    if "--hinglish" in argv:
        i = argv.index("--hinglish")
        res.append(run_language("hinglish", argv[i+1], argv[i+2], argv[i+3], outdir, cfg))
    else:
        res.append(run_language("hinglish", None, None, None, outdir, cfg))

    if "--spanglish" in argv:
        i = argv.index("--spanglish")
        res.append(run_language("spanglish", argv[i+1], argv[i+2], argv[i+3], outdir, cfg))
    else:
        res.append(run_language("spanglish", None, None, None, outdir, cfg))

    summary = pd.concat(res, ignore_index=True)
    (Path(outdir) / "baseline_plus_summary.csv").write_text(summary.to_csv(index=False))
    print("\nSummary:\n", summary[["lang","split","BLEU","chrF","ExactMatch","eq_src","empties","dup_top"]])

if __name__ == "__main__":
    main()


[hinglish-val] n=94 | BLEU=22.68 chrF=36.96 EM=19.15 | pred==src=2 | empty=0 | topdup=2 | ascii=1.000
  FLAGS:
   - ≥1% predictions equal SOURCE (wrong column).
[hinglish-test] n=95 | BLEU=26.70 chrF=41.45 EM=23.16 | pred==src=1 | empty=0 | topdup=2 | ascii=1.000
  FLAGS:
   - ≥1% predictions equal SOURCE (wrong column).
[hinglish] Oracle upper bound BLEU (train-ref -> test-ref): 37.24
[spanglish-val] n=106 | BLEU=1.57 chrF=22.83 EM=0.00 | pred==src=0 | empty=0 | topdup=3 | ascii=0.999
  FLAGS:
   - BLEU≈0 but chrF>20 → often wrong column / copied source / misalignment.
[spanglish-test] n=106 | BLEU=1.44 chrF=23.06 EM=0.00 | pred==src=0 | empty=0 | topdup=3 | ascii=1.000
  FLAGS:
   - BLEU≈0 but chrF>20 → often wrong column / copied source / misalignment.
[spanglish] Oracle upper bound BLEU (train-ref -> test-ref): 2.45

[VERDICT] ❌ Spanglish baseline is inconsistent. See FLAGS above.
Fix order/columns or per-language fit; predictions must be English.

Summary:
         lang split     